# Join-key audit: new sources → dim_country

**What this notebook does.** It checks whether the four new raw sources (CBAM, Open CEDA, OECD maritime, SBTi) can join to `dim_country`, the project's country dimension. `dim_country` has a `census_cty_code` primary key, an `iso3` column (populated for 240 of 241 countries; Kosovo is the exception) and a `cty_name` from the Census Schedule C.

**Why it matters.** Every analysis that uses these sources must link them to countries already in `core`. A silent name mismatch drops rows. This audit makes every mismatch visible before any model is built.

**To run:** set `ANALYST_DATABASE_URL` in `analysis/.env`, then run all cells.

## 1. Setup

In [1]:
import polars as pl

from shockt_analysis import core
from shockt_analysis.settings import get_settings

pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_width_chars(140)

polars.config.Config

In [2]:
with core.connect(get_settings()) as conn:
    dim = core.countries(conn)
    cbam = core.cbam_countries(conn)
    ceda = core.ceda_country_codes(conn)
    oecd = core.oecd_ref_areas(conn)
    sbti = core.sbti_locations(conn)
    loads = core.current_loads(conn, [
        "cbam_default_values", "open_ceda_factors",
        "oecd_maritime_emissions", "sbti_companies",
    ])

n_iso3 = dim.filter(pl.col("iso3").is_not_null()).height
print(f"dim_country: {dim.height} countries, {n_iso3} with ISO3")
print(f"CBAM: {cbam.height} countries, {cbam['n_cn_codes'].sum()} CN-code rows")
print(f"Open CEDA: {ceda.height} country codes")
print(f"OECD: {oecd.height} ref_area values, {oecd['n_obs'].sum()} observations")
print(f"SBTi: {sbti.height} locations, {sbti['n_companies'].sum()} companies")
print()
loads

dim_country: 241 countries, 240 with ISO3
CBAM: 122 countries, 12540 CN-code rows
Open CEDA: 149 country codes
OECD: 162 ref_area values, 8598 observations
SBTi: 113 locations, 15909 companies



dataset,partitions,latest_load,code_versions
str,i64,str,str
"""cbam_default_values""",1,"""2026-10-29 20:48:37.027522+00""","""565063f"""
"""oecd_maritime_emissions""",4,"""2026-10-29 20:49:06.715992+00""","""565063f"""
"""open_ceda_factors""",1,"""2026-10-29 20:48:42.057868+00""","""565063f"""
"""sbti_companies""",1,"""2026-10-29 20:48:50.567017+00""","""565063f"""


## 2. Open CEDA — join on ISO3

Open CEDA has an ISO3-style `country_code`. Join it to `dim_country.iso3`.

In [3]:
ceda_joined = ceda.join(dim, left_on="country_code", right_on="iso3", how="left")
ceda_matched = ceda_joined.filter(pl.col("census_cty_code").is_not_null()).height
ceda_unmatched = ceda_joined.filter(pl.col("census_cty_code").is_null())

match_pct = ceda_matched / ceda.height
print(f"Matched: {ceda_matched} / {ceda.height} ({match_pct:.1%})")
print("Unmatched:")
ceda_unmatched.select("country_code", "country")

Matched: 148 / 149 (99.3%)
Unmatched:


country_code,country
str,str
"""ROW""","""Rest of World"""


Only `ROW` (Rest of World) is unmatched — an aggregate, not a country. **CEDA is clean.**

## 3. OECD — join on ISO3

OECD `ref_area` is ISO3 for countries and group codes for aggregates.

In [4]:
OECD_AGGREGATES = {"W", "OECD", "EU27_2020", "G7", "G20", "W_O"}
oecd_countries = oecd.filter(~pl.col("ref_area").is_in(OECD_AGGREGATES))
oecd_agg = oecd.filter(pl.col("ref_area").is_in(OECD_AGGREGATES))

oecd_joined = oecd_countries.join(
    dim, left_on="ref_area", right_on="iso3", how="left",
)
oecd_matched = oecd_joined.filter(pl.col("census_cty_code").is_not_null()).height
oecd_unmatched = oecd_joined.filter(pl.col("census_cty_code").is_null())

match_pct = oecd_matched / oecd_countries.height
print(f"Aggregates (excluded): {sorted(oecd_agg['ref_area'].to_list())}")
print(f"Country rows: {oecd_countries.height}")
print(f"Matched: {oecd_matched} / {oecd_countries.height} ({match_pct:.1%})")
print("Unmatched:")
oecd_unmatched.select("ref_area", "n_obs")

Aggregates (excluded): ['OECD', 'W', 'W_O']
Country rows: 159
Matched: 156 / 159 (98.1%)
Unmatched:


ref_area,n_obs
str,i64
"""GGY""",45
"""IMN""",45
"""JEY""",33


`GGY` (Guernsey), `IMN` (Isle of Man) and `JEY` (Jersey) are Crown Dependencies without Census Schedule C entries. They are real territories but have no US trade significance. **OECD is clean.**

## 4. CBAM — join on country name

CBAM has no ISO3 code; its `country` column is a name. Try exact matching to `dim_country.cty_name`.

In [5]:
cbam_joined = cbam.join(dim, left_on="country", right_on="cty_name", how="left")
cbam_matched = cbam_joined.filter(pl.col("census_cty_code").is_not_null())
cbam_unmatched = cbam_joined.filter(pl.col("census_cty_code").is_null())

matched_rows = cbam_matched["n_cn_codes"].sum()
total_rows = cbam["n_cn_codes"].sum()

cty_pct = cbam_matched.height / cbam.height
row_pct = matched_rows / total_rows
print(f"Countries matched: {cbam_matched.height} / {cbam.height} ({cty_pct:.1%})")
print(f"CN-code rows matched: {matched_rows} / {total_rows} ({row_pct:.1%})")
gap = total_rows - matched_rows
print(f"\nUnmatched ({cbam_unmatched.height} countries, {gap} rows):")
cbam_unmatched.select("country", "n_cn_codes").sort("n_cn_codes", descending=True)

Countries matched: 104 / 122 (85.2%)
CN-code rows matched: 10052 / 12540 (80.2%)

Unmatched (18 countries, 2488 rows):


country,n_cn_codes
str,i64
"""Iran, Islamic Republic of""",283
"""Korea, Republic of (South Kore…",283
"""Other Countries and Territorie…",283
"""Russian Federation""",283
"""Türkiye""",283
"""United States""",283
"""Viet Nam""",275
"""Myanmar""",259
"""Syria""",62


### Diagnosis

Compare each CBAM name to the Census name it should match.

In [6]:
diagnosis_rows: list[dict[str, object]] = []
for row in cbam_unmatched.iter_rows(named=True):
    name = row["country"]
    keyword = name.split(",")[0].split("(")[0].strip()
    if keyword == "Ivory Coast":
        keyword = "Cote"
    elif keyword == "Other Countries and Territories":
        diagnosis_rows.append({
            "cbam_name": name, "census_name": "(aggregate)",
            "iso3": None, "cause": "aggregate",
        })
        continue
    candidates = dim.filter(
        pl.col("cty_name").str.contains(f"(?i){keyword}"),
    )
    if candidates.height == 1:
        c = candidates.row(0, named=True)
        diagnosis_rows.append({
            "cbam_name": name, "census_name": c["cty_name"],
            "iso3": c["iso3"], "cause": "name variant",
        })
    elif candidates.height > 1:
        for c in candidates.iter_rows(named=True):
            diagnosis_rows.append({
                "cbam_name": name, "census_name": c["cty_name"],
                "iso3": c["iso3"], "cause": "ambiguous",
            })
    else:
        diagnosis_rows.append({
            "cbam_name": name, "census_name": None,
            "iso3": None, "cause": "no match",
        })

diagnosis = pl.DataFrame(diagnosis_rows)
diagnosis

cbam_name,census_name,iso3,cause
str,str,str,str
"""Congo""","""Congo, Republic of the Congo""","""COG""","""ambiguous"""
"""Congo""","""Congo, Democratic Republic of …","""COD""","""ambiguous"""
"""Congo, Democratic Republic of""","""Congo, Republic of the Congo""","""COG""","""ambiguous"""
"""Congo, Democratic Republic of""","""Congo, Democratic Republic of …","""COD""","""ambiguous"""
"""Curaçao""",null,null,"""no match"""
"""Iran, Islamic Republic of""","""Iran""","""IRN""","""name variant"""
"""Ivory Coast""","""Cote d'Ivoire""","""CIV""","""name variant"""
"""Korea, Republic of (South Kore…","""North Korea (Democratic People…","""PRK""","""ambiguous"""
"""Korea, Republic of (South Kore…","""South Korea (Republic of Korea…","""KOR""","""ambiguous"""


Every mismatch except "Other Countries and Territories" (an aggregate) is a name variant of a Census country. The mismatches fall into patterns:

1. **Formal vs common name**: "Russian Federation" vs "Russia", "Viet Nam" vs "Vietnam", "Türkiye" vs "Turkey"
2. **Different qualifier style**: "Korea, Republic of (South Korea)" vs "South Korea (Republic of Korea)"
3. **Census parenthetical**: "Syria" vs "Syria (Syrian Arab Republic)", "Laos" vs "Laos (Lao People's Democratic Republic)"
4. **Different common name**: "Ivory Coast" vs "Cote d'Ivoire", "Myanmar" vs "Burma (Myanmar)"
5. **Accent**: "Curaçao" vs "Curacao"

## 5. SBTi — join on location name

SBTi `location` is a country name. Same approach.

In [7]:
sbti_joined = sbti.join(dim, left_on="location", right_on="cty_name", how="left")
sbti_matched = sbti_joined.filter(pl.col("census_cty_code").is_not_null())
sbti_unmatched = sbti_joined.filter(pl.col("census_cty_code").is_null())

matched_co = sbti_matched["n_companies"].sum()
total_co = sbti["n_companies"].sum()

loc_pct = sbti_matched.height / sbti.height
co_pct = matched_co / total_co
print(f"Locations matched: {sbti_matched.height} / {sbti.height} ({loc_pct:.1%})")
print(f"Companies matched: {matched_co} / {total_co} ({co_pct:.1%})")
gap = total_co - matched_co
print(f"\nUnmatched ({sbti_unmatched.height} locations, {gap} companies):")
sbti_unmatched.select("location", "n_companies").sort(
    "n_companies", descending=True,
)

Locations matched: 96 / 113 (85.0%)
Companies matched: 11307 / 15909 (71.1%)

Unmatched (17 locations, 4602 companies):


location,n_companies
str,i64
"""United Kingdom of Great Britai…",1976
"""Germany""",984
"""Netherlands (Kingdom of the)""",547
"""Denmark""",380
"""Taiwan, Province of China""",234
"""Türkiye""",139
"""Republic of Korea""",129
"""China, Hong Kong Special Admin…",120
"""Viet Nam""",44


In [8]:
sbti_diag_rows: list[dict[str, object]] = []
for row in sbti_unmatched.iter_rows(named=True):
    name = row["location"]
    keyword = name.split(",")[0].split("(")[0].strip()
    if keyword == "Republic of Korea":
        keyword = "South Korea"
    elif keyword == "Czechia":
        keyword = "Czech"
    candidates = dim.filter(
        pl.col("cty_name").str.contains(f"(?i){keyword}"),
    )
    if candidates.height == 1:
        c = candidates.row(0, named=True)
        sbti_diag_rows.append({
            "sbti_location": name,
            "census_name": c["cty_name"], "iso3": c["iso3"],
        })
    elif candidates.height > 1:
        for c in candidates.iter_rows(named=True):
            sbti_diag_rows.append({
                "sbti_location": name,
                "census_name": c["cty_name"], "iso3": c["iso3"],
            })
    else:
        sbti_diag_rows.append({
            "sbti_location": name,
            "census_name": None, "iso3": None,
        })

sbti_diag = pl.DataFrame(sbti_diag_rows)
sbti_diag

sbti_location,census_name,iso3
str,str,str
"""Bolivia (Plurinational State o…","""Bolivia""","""BOL"""
"""China, Hong Kong Special Admin…","""China""","""CHN"""
"""China, Macao Special Administr…","""China""","""CHN"""
"""Czechia""","""Czech Republic""","""CZE"""
"""Denmark""","""Denmark, except Greenland""","""DNK"""
"""Germany""","""Germany (Federal Republic of G…","""DEU"""
"""Jersey""",null,null
"""Myanmar""","""Burma (Myanmar)""","""MMR"""
"""Netherlands (Kingdom of the)""","""Netherlands""","""NLD"""


All 17 SBTi mismatches are name variants. The biggest gaps are the UK (1,976 companies), Germany (984) and Netherlands (547) — countries whose Census names include formal qualifiers.

## 6. Summary

All values computed from the cells above.

In [9]:
summary = pl.DataFrame({
    "source": ["Open CEDA", "OECD maritime", "CBAM", "SBTi"],
    "join_key": [
        "country_code (ISO3)", "ref_area (ISO3)",
        "country (name)", "location (name)",
    ],
    "join_target": [
        "dim_country.iso3", "dim_country.iso3",
        "dim_country.cty_name", "dim_country.cty_name",
    ],
    "total_keys": [
        ceda.height, oecd_countries.height,
        cbam.height, sbti.height,
    ],
    "matched": [
        ceda_matched, oecd_matched,
        cbam_matched.height, sbti_matched.height,
    ],
    "unmatched_cause": [
        "1 aggregate (ROW)",
        "3 Crown Dependencies",
        "17 name variants + 1 aggregate",
        "17 name variants",
    ],
})
summary = summary.with_columns(
    (pl.col("matched") / pl.col("total_keys") * 100)
    .round(1)
    .alias("match_pct"),
)
summary

source,join_key,join_target,total_keys,matched,unmatched_cause,match_pct
str,str,str,i64,i64,str,f64
"""Open CEDA""","""country_code (ISO3)""","""dim_country.iso3""",149,148,"""1 aggregate (ROW)""",99.3
"""OECD maritime""","""ref_area (ISO3)""","""dim_country.iso3""",159,156,"""3 Crown Dependencies""",98.1
"""CBAM""","""country (name)""","""dim_country.cty_name""",122,104,"""17 name variants + 1 aggregate""",85.2
"""SBTi""","""location (name)""","""dim_country.cty_name""",113,96,"""17 name variants""",85.0


In [10]:
findings = [
    "ISO3 is the reliable join key. dim_country has it for "
    "240 of 241 countries (Kosovo excluded).",
    f"Open CEDA joins on ISO3 with {ceda_matched}/{ceda.height} "
    "matched. Only 'ROW' (Rest of World) is unmatched.",
    f"OECD joins on ISO3 with {oecd_matched}/{oecd_countries.height} "
    "countries matched. Guernsey, Isle of Man and Jersey "
    "are not in the Census Schedule C.",
    f"CBAM joins on name with {cbam_matched.height}/{cbam.height} "
    f"countries matched ({matched_rows}/{total_rows} rows, "
    f"{row_pct:.0%}). 17 real countries fail due to name "
    "variants; 1 aggregate.",
    f"SBTi joins on name with {sbti_matched.height}/{sbti.height} "
    f"locations matched ({matched_co}/{total_co} companies, "
    f"{co_pct:.0%}). 17 locations fail; UK + Germany + "
    "Netherlands alone account for 3,507 companies.",
    "All name mismatches follow five patterns: formal vs common "
    "name, qualifier style, Census parentheticals, different "
    "common name (Ivory Coast / Cote d'Ivoire, Myanmar / Burma), "
    "and accents (Curaçao / Curacao).",
    "Resolution: CEDA and OECD staging models join on ISO3. "
    "CBAM and SBTi staging models need a name-to-ISO3 mapping "
    "derived from a published source (ISO 3166), not hand-typed.",
]
for i, f in enumerate(findings, 1):
    print(f"{i}. {f}")

1. ISO3 is the reliable join key. dim_country has it for 240 of 241 countries (Kosovo excluded).
2. Open CEDA joins on ISO3 with 148/149 matched. Only 'ROW' (Rest of World) is unmatched.
3. OECD joins on ISO3 with 156/159 countries matched. Guernsey, Isle of Man and Jersey are not in the Census Schedule C.
4. CBAM joins on name with 104/122 countries matched (10052/12540 rows, 80%). 17 real countries fail due to name variants; 1 aggregate.
5. SBTi joins on name with 96/113 locations matched (11307/15909 companies, 71%). 17 locations fail; UK + Germany + Netherlands alone account for 3,507 companies.
6. All name mismatches follow five patterns: formal vs common name, qualifier style, Census parentheticals, different common name (Ivory Coast / Cote d'Ivoire, Myanmar / Burma), and accents (Curaçao / Curacao).
7. Resolution: CEDA and OECD staging models join on ISO3. CBAM and SBTi staging models need a name-to-ISO3 mapping derived from a published source (ISO 3166), not hand-typed.
